[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1jrmm8eTLsHPMFF6XEkBq5GkhS51gtuqt?usp=sharing)

# 📉 Análisis Financiero: El Costo de Oportunidad del Tabaquismo

**Autor:** Eliuth Misraim Rojas Villavicencio  
**Fecha:** Diciembre 2025  
**Contexto:** Portafolio de Data Science / Salud Pública  
**Herramientas:** Python + Plotly (Visualización Interactiva)

### 🎯 Objetivo
Este notebook tiene como objetivo modelar matemáticamente el impacto patrimonial de fumar a largo plazo. No solo analizaremos el **gasto directo** (dinero quemado), sino el **costo de oportunidad**: ¿Cuánto dinero tendría esta persona si hubiera invertido ese capital en un instrumento financiero compuesto (S&P 500 / CETES)?

> *"El interés compuesto es la octava maravilla del mundo. Quien lo entiende, lo gana; quien no, lo paga."* - Albert Einstein

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# Configuración de renderizado (necesario para ver Plotly en GitHub/Colab a veces)
pio.renderers.default = 'colab'

## 1. Fundamentos Matemáticos y Supuestos

Para cuantificar la pérdida patrimonial, utilizamos el modelo de **Valor Futuro de una Anualidad Creciente**. A diferencia de una calculadora simple, este modelo considera dos factores dinámicos clave:

1.  **Inflación del Producto ($g$):** El precio del tabaco aumenta anualmente.
2.  **Tasa de Retorno ($r$):** El rendimiento de invertir ese dinero (ej. S&P 500 o CETES).

La lógica subyacente se basa en la acumulación de capital bajo la fórmula de interés compuesto:

$$FV = \sum_{t=1}^{n} P_t \times (1 + r)^{n-t}$$

Donde:
* $FV$: Valor Futuro (Patrimonio Final)
* $P_t$: Gasto anual en el año $t$ (ajustado por inflación)
* $r$: Tasa de interés anual (Retorno de Inversión)
* $n$: Horizonte de tiempo en años

### Parámetros de la Simulación
* **Consumo:** Media cajetilla diaria (10 cigarros).
* **Costo Base:** $75.00 MXN por cajetilla.
* **Horizonte:** 20 Años.

In [3]:
# --- PARÁMETROS DEL MODELO ---

# Variables de Consumo
CIGARROS_DIARIOS = 10        
PRECIO_CAJETILLA = 75.00     # Precio 2025 (MXN)
CIGARROS_PAQUETE = 20

# Variables Macroeconómicas
ANIOS_PROYECCION = 20        
TASA_RETORNO_ANUAL = 0.10    # 10% (Media histórica conservadora)
INFLACION_TABACO = 0.05      # 5% Aumento anual de precio

# Cálculo del Gasto Base (Año 0)
gasto_diario = (CIGARROS_DIARIOS / CIGARROS_PAQUETE) * PRECIO_CAJETILLA
gasto_anual_inicial = gasto_diario * 365

print(f"💰 Gasto Diario Base: ${gasto_diario:.2f} MXN")
print(f"🔥 Gasto Anual Inicial: ${gasto_anual_inicial:,.2f} MXN")

💰 Gasto Diario Base: $37.50 MXN
🔥 Gasto Anual Inicial: $13,687.50 MXN


## 2. Motor de Simulación (Iteración Anual)

Ejecutamos un algoritmo iterativo para proyectar dos escenarios futuros paralelos:

* 🔴 **Escenario A (Gasto Irrecuperable):** Acumulación lineal del gasto, ajustado por la inflación del tabaco. Representa la pérdida neta de riqueza.
* 🟢 **Escenario B (Inversión Pasiva):** Reinversión automática de los flujos de efectivo. Se aprovecha el efecto exponencial del interés compuesto sobre el capital acumulado.

In [4]:
def generar_proyeccion_financiera(anios, aporte_inicial, inflacion, tasa_interes):
    registros = []
    
    saldo_quemado = 0      
    saldo_invertido = 0    
    aporte_actual = aporte_inicial
    
    for anio in range(1, anios + 1):
        # 1. Escenario Pérdida (Suma simple del gasto)
        saldo_quemado += aporte_actual
        
        # 2. Escenario Inversión (Interés Compuesto)
        # (Saldo anterior + rendimientos) + Aportación nueva
        saldo_invertido = (saldo_invertido * (1 + tasa_interes)) + aporte_actual
        
        # Cálculo del Costo de Oportunidad (Brecha)
        brecha = saldo_invertido - saldo_quemado
        
        registros.append({
            "Año": anio,
            "Gasto Anual": aporte_actual,
            "Dinero Quemado (Pérdida)": saldo_quemado,
            "Patrimonio Invertido (Ganancia)": saldo_invertido,
            "Costo de Oportunidad": brecha
        })
        
        # Ajuste inflacionario para el siguiente ciclo
        aporte_actual = aporte_actual * (1 + inflacion)
        
    return pd.DataFrame(registros)

# Ejecución
df_resultados = generar_proyeccion_financiera(
    ANIOS_PROYECCION, 
    gasto_anual_inicial, 
    INFLACION_TABACO, 
    TASA_RETORNO_ANUAL
)
# Ejecución
df_resultados = generar_proyeccion_financiera(
    ANIOS_PROYECCION, 
    gasto_anual_inicial, 
    INFLACION_TABACO, 
    TASA_RETORNO_ANUAL
)

# --- MEJORA VISUAL DE LA TABLA ---

# Seleccionamos los últimos 5 años para mostrar
df_view = df_resultados.tail(100)

# Aplicamos estilos
# 1. Definimos el formato de moneda (Signo $, comas y 2 decimales)
format_dict = {
    'Gasto Anual': '${:,.2f}',
    'Dinero Quemado (Pérdida)': '${:,.2f}',
    'Patrimonio Invertido (Ganancia)': '${:,.2f}',
    'Costo de Oportunidad': '${:,.2f}'
}

# 2. Renderizamos la tabla con estilos
tabla_estilizada = df_view.style.format(format_dict)\
    .background_gradient(subset=['Patrimonio Invertido (Ganancia)'], cmap='Greens')\
    .background_gradient(subset=['Dinero Quemado (Pérdida)'], cmap='Reds')\
    .hide(axis='index') # Ocultamos el índice numérico de pandas que no aporta valor

# Mostrar la tabla
tabla_estilizada

Año,Gasto Anual,Dinero Quemado (Pérdida),Patrimonio Invertido (Ganancia),Costo de Oportunidad
1,"$13,687.50","$13,687.50","$13,687.50",$0.00
2,"$14,371.88","$28,059.38","$29,428.12","$1,368.75"
3,"$15,090.47","$43,149.84","$47,461.41","$4,311.56"
4,"$15,844.99","$58,994.84","$68,052.54","$9,057.70"
5,"$16,637.24","$75,632.08","$91,495.03","$15,862.96"
6,"$17,469.10","$93,101.18","$118,113.64","$25,012.46"
7,"$18,342.56","$111,443.74","$148,267.57","$36,823.82"
8,"$19,259.69","$130,703.43","$182,354.01","$51,650.58"
9,"$20,222.67","$150,926.10","$220,812.08","$69,885.98"
10,"$21,233.80","$172,159.90","$264,127.09","$91,967.19"


## 3. Visualización Interactiva del Costo de Oportunidad

A continuación, se presenta la proyección gráfica. El área sombreada entre las curvas representa el **valor económico destruido** por el hábito.

> *Nota: Interactúe con el gráfico (Zoom/Hover) para inspeccionar los valores específicos de cada año.*

In [5]:
# Crear Figura Interactiva
fig = go.Figure()

# Traza 1: Dinero Quemado
fig.add_trace(go.Scatter(
    x=df_resultados['Año'], 
    y=df_resultados['Dinero Quemado (Pérdida)'],
    mode='lines+markers',
    name='Dinero Quemado (Pérdida Neta)',
    line=dict(color='#e74c3c', width=2, dash='dash'),
    hovertemplate='<b>Año %{x}</b><br>Pérdida Acumulada: $%{y:,.0f}<extra></extra>'
))

# Traza 2: Inversión (Interés Compuesto)
fig.add_trace(go.Scatter(
    x=df_resultados['Año'], 
    y=df_resultados['Patrimonio Invertido (Ganancia)'],
    mode='lines+markers',
    name='Patrimonio Potencial (Inversión)',
    line=dict(color='#27ae60', width=4),
    fill='tonexty', # Relleno dinámico hasta la curva anterior
    fillcolor='rgba(39, 174, 96, 0.15)',
    hovertemplate='<b>Año %{x}</b><br>Patrimonio Potencial: $%{y:,.0f}<extra></extra>'
))

# Configuración del Layout (Estética Profesional)
monto_final = df_resultados.iloc[-1]['Patrimonio Invertido (Ganancia)']

fig.update_layout(
    title=dict(
        text=f'<b>Análisis de Patrimonio: Costo de Oportunidad a {ANIOS_PROYECCION} Años</b>',
        font=dict(size=18, color='#2c3e50')
    ),
    xaxis_title='Tiempo Transcurrido (Años)',
    yaxis_title='Monto Acumulado (MXN)',
    template='plotly_white',
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Callout (Anotación) del valor final
fig.add_annotation(
    x=ANIOS_PROYECCION,
    y=monto_final,
    text=f"<b>${monto_final:,.0f} MXN</b>",
    showarrow=True,
    arrowhead=2,
    ax=-40,
    ay=-40,
    font=dict(size=14, color="#27ae60")
)

fig.show()

## 4. Análisis de Retorno de Inversión (ROI)

Finalmente, evaluamos la rentabilidad del curso para dejar de fumar. Si consideramos el costo del curso como una inversión inicial ($I_0$), calculamos el **Periodo de Recuperación (Payback Period)** basándonos en el flujo de ahorro mensual generado.

$$Payback = \frac{Inversion Inicial}{Ahorro Mensual}$$

In [8]:
# --- ROI DEL TRATAMIENTO ---
COSTO_CURSO = 3000.00  # MXN

ahorro_mensual_promedio = df_resultados.iloc[0]['Gasto Anual'] / 12
meses_recuperacion = COSTO_CURSO / ahorro_mensual_promedio

# Visualización tipo Indicador (Gauge)
fig_roi = go.Figure(go.Indicator(
    mode = "number+gauge+delta",
    value = meses_recuperacion,
    domain = {'x': [0, 1], 'y': [0, 1]},
    title = {'text': "<b>Meses para recuperar la Inversión</b><br><span style='font-size:0.8em;color:gray'>Basado en ahorro directo de cigarros</span>"},
    delta = {'reference': 6, 'increasing': {'color': "#e74c3c"}, 'decreasing': {'color': "#27ae60"}},
    gauge = {
        'axis': {'range': [None, 12], 'tickwidth': 1},
        'bar': {'color': "#2c3e50"},
        'bgcolor': "white",
        'steps': [
            {'range': [0, 3], 'color': "rgba(39, 174, 96, 0.3)"},  # Verde claro (Excelente)
            {'range': [3, 6], 'color': "rgba(241, 196, 15, 0.3)"}, # Amarillo (Bueno)
            {'range': [6, 12], 'color': "rgba(231, 76, 60, 0.3)"}  # Rojo (Lento)
        ]}
))

fig_roi.update_layout(height=350)
fig_roi.show()

print(f"✅ Conclusión: Dejando de fumar hoy, el curso se paga solo en {meses_recuperacion:.1f} meses.")

✅ Conclusión: Dejando de fumar hoy, el curso se paga solo en 2.6 meses.


## 4. Simulador Interactivo de ROI (Retorno de Inversión)

A continuación, presentamos una herramienta dinámica. Utiliza el **control deslizante (slider)** para simular diferentes precios del curso o tratamiento.

El indicador calculará automáticamente:
1.  **Meses de Recuperación:** Tiempo necesario para amortizar la inversión basándose en el ahorro de tabaco.
2.  **Calificación Financiera:**
    * 🟢 **Excelente:** Recuperación en menos de 3 meses.
    * 🟡 **Buena:** Recuperación entre 3 y 6 meses.
    * 🔴 **Lenta:** Más de 6 meses.

In [7]:
from ipywidgets import interact, IntSlider
from IPython.display import display, clear_output # Importamos para limpiar pantalla
import plotly.graph_objects as go

# Calculamos el ahorro mensual base (fijo) del primer año
AHORRO_MENSUAL_BASE = df_resultados.iloc[0]['Gasto Anual'] / 12

def actualizar_medidor(costo_curso):
    """
    Recalcula el ROI y regenera la gráfica
    """
    meses_recuperacion = costo_curso / AHORRO_MENSUAL_BASE
    
    # Crear el indicador (Gauge)
    fig = go.Figure(go.Indicator(
        mode = "number+gauge+delta",
        value = meses_recuperacion,
        domain = {'x': [0, 1], 'y': [0, 1]},
        title = {'text': f"<b>Meses para recuperar ${costo_curso:,.0f}</b><br><span style='font-size:0.8em;color:gray'>Ahorrando ${AHORRO_MENSUAL_BASE:,.0f}/mes</span>"},
        delta = {'reference': 6, 'increasing': {'color': "#e74c3c"}, 'decreasing': {'color': "#27ae60"}},
        gauge = {
            'axis': {'range': [None, 12], 'tickwidth': 1},
            'bar': {'color': "#2c3e50"},
            'bgcolor': "white",
            'steps': [
                {'range': [0, 3], 'color': "rgba(39, 174, 96, 0.4)"},  # Verde
                {'range': [3, 6], 'color': "rgba(241, 196, 15, 0.4)"}, # Amarillo
                {'range': [6, 12], 'color': "rgba(231, 76, 60, 0.4)"}  # Rojo
            ],
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': meses_recuperacion}
        }
    ))
    
    fig.update_layout(height=400, margin=dict(t=50, b=0, l=0, r=0))
    
    # TRUCO: Limpiamos la salida anterior para evitar que se acumulen gráficas
    # o que no se refresque
    clear_output(wait=True) 
    
    # Mostramos la nueva figura
    fig.show()

# Creamos el slider manualmente para tener control
slider = IntSlider(
    min=1000, 
    max=15000, 
    step=500, 
    value=6000, 
    description='Inversión ($):',
    continuous_update=False # Importante: actualiza solo al soltar el mouse para que no "parpadee"
)

# Ejecutamos la interacción
interact(actualizar_medidor, costo_curso=slider);

interactive(children=(IntSlider(value=6000, continuous_update=False, description='Inversión ($):', max=15000, …